<a href="https://colab.research.google.com/github/galvanizer10/05_Rasters_and_Satellite_Imagery-/blob/master/05b_Floods_In_Iraq_ipyb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
ee.Authenticate()

In [ ]:
ee.Initialize(project = 'agile-aleph-503014-h5')

In [ ]:
pre_event_beg = '2022-07-14'
pre_event_final = '2022-07-31'

post_event_beg = '2022-10-01'
post_event_final = '2022-10-15'

In [ ]:
fao_gaul = ee.FeatureCollection('FAO/GAUL/2015/level1')
pakistan = fao_gaul.filter(ee.Filter.eq('ADM0_NAME', 'Pakistan'))
sindh = pakistan.filter(ee.Filter.eq('ADM1_NAME', 'Sindh'))
roi = sindh.geometry()

In [ ]:
Map = geemap.Map(center=[26.5, 68.5], zoom=9)
Map.addLayer(roi, {'color': 'gray'}, 'Study Area')
Map

Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
# #changing to geo data frame
# gpd_dataframe = geemap.ee_to_df(roi)

In [ ]:
S1collection_1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
                   .filter(ee.Filter.eq('instrumentMode', 'IW')) \
                   .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
                   .filterMetadata('resolution_meters', 'equals' , 10) \
                   .filterBounds(roi) \
                   .filterDate(pre_event_beg, pre_event_final) \
                   .select('VV')

In [ ]:
S1collection_2 = ee.ImageCollection('COPERNICUS/S1_GRD') \
                   .filter(ee.Filter.eq('instrumentMode', 'IW')) \
                   .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
                   .filterMetadata('resolution_meters', 'equals' , 10) \
                   .filterBounds(roi) \
                   .filterDate(post_event_beg, post_event_final) \
                   .select('VV')

In [ ]:
S1_1 = S1collection_1.mosaic().clip(roi)
S1_2 = S1collection_2.mosaic().clip(roi)

print(type(S1_1))

<class 'ee.image.Image'>


In [ ]:
SMOOTHING_RADIUS = 30

pre_event_filtered = S1_1.focal_mean(SMOOTHING_RADIUS, 'circle', 'meters')
post_event_filtered = S1_2.focal_mean(SMOOTHING_RADIUS, 'circle', 'meters')

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(S1_1, {'min':-25, 'max':-5}, 'S1-1')
Map.addLayer(S1_2, {'min':-25, 'max':-5}, 'S1-2')
Map.addLayer(pre_event_filtered, {'min':-25, 'max':-5}, 'S1-1-Filt')
Map.addLayer(post_event_filtered, {'min':-25, 'max':-5}, 'S1-2-Filt')
Map

S1_VV_RGB = ee.Image.cat(pre_event_filtered.select('VV'), post_event_filtered.select('VV'), pre_event_filtered.select('VV'))

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(S1_VV_RGB, {'min':-25, 'max':0}, 'S1-VV-RGB')
Map


Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
differenceVV = post_event_filtered.select('VV').divide(pre_event_filtered.select('VV'))

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(differenceVV, {'min': 0, 'max':2}, 'difference VV filtered')

UPPER_THRESHOLD = 1.15
LOWER_THRESHOLD = 0.7

inundation1 = differenceVV.gt(UPPER_THRESHOLD).Or(differenceVV.lt(LOWER_THRESHOLD))

Map.addLayer(inundation1.updateMask(inundation1),
             {'palette':"GnBu"},'Flooded Areas - RAW')
Map

Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
connections = inundation1.connectedPixelCount()
inundation2 = inundation1.updateMask(connections.gte(8))

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(roi, {'color': 'gray'}, 'Study Area')
Map.addLayer(inundation2.updateMask(inundation2), {'palette':"GnBu"},'Flooded Areas - non-sparse')
Map

Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
srtm = ee.Image('USGS/SRTMGL1_003')
terrain = ee.Algorithms.Terrain(srtm)
slope = terrain.select('slope')

inundation3 = inundation2.updateMask(slope.lt(5))

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(srtm, {'min':0, 'max':1000}, 'SRTM')
Map.addLayer(roi, {'color': 'gray'}, 'Study Area')
Map.addLayer(inundation3.updateMask(inundation3),{'palette':"GnBu"},'Flooded Areas - slope-adjusted')
Map

Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
inundation_area_pixel = inundation3.multiply(ee.Image.pixelArea())

inundation_stats = inundation_area_pixel.reduceRegion(
                                         reducer= ee.Reducer.sum(),
                                         geometry= roi,
                                         scale= 10,
                                         bestEffort= True)
print(inundation_stats.getInfo())

{'VV': 40110403131.29549}


In [ ]:
inundation_area_ha = inundation_stats \
                          .getNumber("VV") \
                          .divide(10000) \
                          .round()
print(inundation_area_ha.getInfo())

4011040


In [ ]:
def maskS2clouds(image):
    qa = image.select('QA60')

    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11

    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
            .And(qa.bitwiseAnd(cirrusBitMask).eq(0))

    return image.updateMask(mask).divide(10000)

vizParams = {
  'bands': ['B4', 'B3', 'B2'],
  'min': 0,
  'max': 0.4,
  'gamma': [0.98, 1.1, 1]
}

In [ ]:
post_floods = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
               .filterDate('2017-10-01', '2018-10-31') \
               .filterBounds(roi) \
               .map(maskS2clouds)

medianpixels1 = post_floods.median()
medianpixelsclipped1 = medianpixels1.clip(roi)

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(medianpixelsclipped1, vizParams, 'Post-Floods S2')
Map

Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

# Mapping Pakistan Floods

In [ ]:
pre_floods = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
              .filterDate('2015-08-31', '2017-08-31') \
              .filterBounds(roi) \
              .map(maskS2clouds)

medianpixels2 = pre_floods.median()
medianpixelsclipped2 = medianpixels2.clip(roi)

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(medianpixelsclipped2, vizParams, 'Pre-Maria S2')
Map

Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
post_ndvi = medianpixels1.normalizedDifference(['B8', 'B4'])


visParams_ndvi = {'min': -0.2, 'max': 0.8, 'palette': ['blue', 'white', 'yellow', 'green']}

post_ndvi_clip = post_ndvi.clip(roi)

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(post_ndvi_clip, visParams_ndvi, 'Post-Maria NDVI')

In [ ]:
pre_ndvi = medianpixels2.normalizedDifference(['B8', 'B4'])
pre_ndvi_clip = pre_ndvi.clip(roi)

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(pre_ndvi_clip, visParams_ndvi, 'Pre-Maria NDVI')


In [ ]:
NDVI_diff = post_ndvi_clip.subtract(pre_ndvi_clip)

Map = geemap.Map(center=[26.5, 68.5], zoom = 8)
Map.addLayer(NDVI_diff, {'min': -0.2, 'max': 0.2, 'palette': ['red', 'yellow', 'green']}, 'NDVI Difference')
Map

Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

# Notebook 8
